# Lean-15 : Le Theoreme de Kochen-Specker (Cabello 18 vecteurs) + Free Will Theorem

**Navigation** : [<< Lean-14 Conway Tribute](Lean-14b-Conway-Game-of-Life-Lean.ipynb) | [Index](README.md)

**Kernel** : Python 3 (illustrations + verifications combinatoires) + Lean 4 (WSL) pour les sections 6-7

---

## Introduction

En 1967, Simon Kochen et Ernst Specker publient un theoreme qui exclut une large classe d'interpretations de la mecanique quantique : les **theories a variables cachees non-contextuelles**. La preuve originale utilise **117 vecteurs** de R^3. Plusieurs raffinements suivent (33 vecteurs par Peres 1991, 20 par Kernaghan 1994) jusqu'a la preuve la plus compacte connue avec **18 vecteurs de R^4** par Cabello, Estebaranz et Garcia-Alcaine en 1996. C'est cette derniere qui sert de noyau combinatoire au theoreme de libre arbitre de Conway-Kochen (2006).

Ce notebook couvre les **Piliers 1 et 3** de l'Epic #1651 (Theoreme de Libre Arbitre Conway-Kochen). Il presente :
1. La question : peut-on attribuer une valeur 0/1 aux observables quantiques de maniere non-contextuelle ?
2. La structure : 18 vecteurs de R^4 en 9 bases orthogonales, chacun appartenant a exactement 2 bases
3. L'argument de parite : 9 (impair) vs 2k (pair) -- contradiction
4. Verification computationnelle : aucune coloration valide n'existe sur 2^18 essais
5. Le port Lean 4 : `KochenSpecker.lean` (Pilier 1, PROUVE) + `FreeWillTheorem.lean` (Pilier 2, PROUVE)
6. Verification `lake build` : 0 sorry dans les 2 Piliers (KochenSpecker + FreeWillTheorem)

### Prerequis
- Notions d'algebre lineaire (R^4, bases orthogonales, produit scalaire)
- Notebooks Lean-1 a Lean-7 pour les aspects formels (recommande)
- Lean-12 pour le pattern Lake build + WSL

### Duree estimee : 75 minutes

## 1. La Question de Kochen-Specker

### 1.1 Le contexte physique

En mecanique quantique, mesurer une grandeur revient a projeter l'etat du systeme sur une famille de vecteurs orthonormaux. Pour un systeme de spin-1, mesurer le carre du spin selon trois axes mutuellement orthogonaux (en 3D) ou quatre axes mutuellement orthogonaux (en 4D pour les paires entrelacees) donne toujours un seul resultat 1 et le reste 0.

**Variable cachee non-contextuelle (NCHV)** : une theorie qui attribue a chaque observable une valeur predeterminee, **independante du contexte de mesure** (i.e. des autres observables mesures simultanement).

Pour les observables associes a des directions vectorielles, cela revient a une fonction $f : \text{vecteurs} \to \{0, 1\}$ telle que dans **toute base orthogonale**, exactement un vecteur recoit la valeur 1.

### 1.2 Le theoreme

**Theoreme (Kochen-Specker 1967, version Cabello et al. 1996)** : Il existe un ensemble fini de vecteurs de R^4 -- precisement 18 vecteurs organises en 9 bases orthogonales -- tel qu'**aucune** fonction $f$ a valeurs dans $\{0, 1\}$ ne peut satisfaire la contrainte "un seul 1 par base".

**Consequence** : aucune theorie a variables cachees non-contextuelle ne peut reproduire les predictions de la mecanique quantique. Toute attribution de valeurs predeterminees doit **dependre du contexte** ou abandonner la realisme local.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from itertools import product
from collections import Counter

np.set_printoptions(suppress=True, precision=4)
print('Setup ok : numpy', np.__version__)

Setup ok : numpy 2.3.4


## 2. Les 18 Vecteurs de Cabello

Les 18 vecteurs de la preuve Cabello et al. (1996) sont enumeres ci-dessous. Ils sont initialement donnes a normalisation pres -- les coordonnees brutes (entieres) suffisent pour les arguments combinatoires d'orthogonalite (le produit scalaire vaut 0 si et seulement si la version normalisee est orthogonale).

**Source** : Cabello, Estebaranz, Garcia-Alcaine, "Bell-Kochen-Specker theorem: A proof with 18 vectors", Phys. Lett. A 212 (1996), 183-187. Table reproduite dans Wikipedia, section Overview.

In [2]:
# Les 18 vecteurs distincts (coordonnees brutes, R^4)
vectors = {
    0:  ( 0,  0,  0,  1),  # v0
    1:  ( 0,  0,  1,  0),  # v1
    2:  ( 1,  1,  0,  0),  # v2
    3:  ( 1, -1,  0,  0),  # v3
    4:  ( 0,  1,  0,  0),  # v4
    5:  ( 1,  0,  1,  0),  # v5
    6:  ( 1,  0, -1,  0),  # v6
    7:  ( 1, -1,  1, -1),  # v7
    8:  ( 1, -1, -1,  1),  # v8
    9:  ( 0,  0,  1,  1),  # v9
    10: ( 1,  1,  1,  1),  # v10
    11: ( 0,  1,  0, -1),  # v11
    12: ( 1,  0,  0,  1),  # v12
    13: ( 1,  0,  0, -1),  # v13
    14: ( 0,  1, -1,  0),  # v14
    15: ( 1,  1, -1,  1),  # v15
    16: ( 1,  1,  1, -1),  # v16
    17: (-1,  1,  1,  1),  # v17
}

# Verifier l'unicite (18 vecteurs distincts apres deduplication)
unique_vecs = {v: k for k, v in vectors.items()}
assert len(unique_vecs) == 18, f'Doublons detectes : {18 - len(unique_vecs)}'
print(f'18 vecteurs distincts : OK')
print(f'Norme L2 (au carre) par vecteur : {sorted(set(sum(x*x for x in v) for v in vectors.values()))}')

18 vecteurs distincts : OK
Norme L2 (au carre) par vecteur : [1, 2, 4]


### Interpretation : les 18 vecteurs

Les 18 vecteurs se groupent en 3 classes selon leur norme :

| Type | Exemple | Norme^2 | Nb vecteurs |
|------|---------|---------|-------------|
| Standards (1 coord) | (0,0,0,1) | 1 | 2 |
| Plans (2 coords +/-) | (1,1,0,0), (1,-1,0,0) | 2 | 8 |
| Quadruplets (4 coords +/-) | (1,1,1,1), (1,-1,1,-1) | 4 | 8 |

Tous deviennent unitaires apres normalisation. L'orthogonalite (produit scalaire = 0) n'est pas affectee par la normalisation.

## 3. Les 9 Bases Orthogonales

Les 18 vecteurs s'organisent en 9 bases orthogonales ("contextes") de 4 vecteurs chacune. Chaque base correspond a une experience de mesure simultanee compatible.

Notation : `Ck = (i, j, k, l)` signifie que la k-ieme base contient les vecteurs d'indices i, j, k, l (dans l'ordre du tableau Cabello original).

In [3]:
# Les 9 contextes (bases orthogonales) -- chaque entree liste 4 indices de vecteurs
contexts = [
    ( 0,  1,  2,  3),  # C0
    ( 0,  4,  5,  6),  # C1
    ( 7,  8,  2,  9),  # C2
    ( 7, 10,  6, 11),  # C3
    ( 1,  4, 12, 13),  # C4
    ( 8, 10, 13, 14),  # C5
    (15, 16,  3,  9),  # C6
    (15, 17,  5, 11),  # C7
    (16, 17, 12, 14),  # C8
]

def dot(u, v):
    return sum(a * b for a, b in zip(u, v))

# Verifier que chaque contexte forme 4 vecteurs mutuellement orthogonaux
ortho_ok = True
for k, ctx in enumerate(contexts):
    pairs = [(i, j) for idx_i, i in enumerate(ctx) for j in ctx[idx_i+1:]]
    for i, j in pairs:
        d = dot(vectors[i], vectors[j])
        if d != 0:
            print(f'  C{k} : v{i} . v{j} = {d} (non-orthogonal !)')
            ortho_ok = False
    if ortho_ok:
        pass

print(f'\n9 contextes, 6 paires par contexte = 54 paires verifiees')
print(f'Toutes orthogonales : {ortho_ok}')


9 contextes, 6 paires par contexte = 54 paires verifiees
Toutes orthogonales : True


### Interpretation : verification d'orthogonalite

Chaque base C0-C8 contient 4 vecteurs mutuellement orthogonaux. Cela donne 6 paires par base (C(4,2) = 6) et 54 paires au total a verifier. La verification renvoie `True` : la table Cabello respecte effectivement la contrainte geometrique d'orthogonalite, prerequis pour interpreter chaque base comme une mesure quantique compatible.

## 4. La Structure d'Overlap (chaque vecteur dans 2 bases)

La cle combinatoire du theoreme est que les 18 vecteurs et les 9 bases sont relies par une **structure d'overlap** : chaque vecteur appartient a exactement 2 des 9 bases.

Pourquoi ? Une base a 4 vecteurs, donc 9 bases x 4 = 36 "slots". Si chaque vecteur etait dans une seule base, il faudrait 36 vecteurs distincts -- on n'en a que 18. Donc le ratio est 36/18 = 2 en moyenne. La table Cabello realise ce ratio **uniformement** : exactement 2 pour chaque vecteur, ni 1, ni 3.

In [4]:
# Compter les occurrences de chaque vecteur dans les 9 contextes
occurrences = Counter()
for ctx in contexts:
    for i in ctx:
        occurrences[i] += 1

# Verifier l'invariant : chaque vecteur apparait exactement 2 fois
print(f'{"Vec":<5} | {"#bases":<8} | {"Bases contenant ce vecteur"}')
print('-' * 65)
all_two = True
for v in range(18):
    bases = [k for k, ctx in enumerate(contexts) if v in ctx]
    print(f'v{v:<4} | {occurrences[v]:<8} | C{bases[0]} et C{bases[1]}' if len(bases)==2 else f'v{v:<4} | {occurrences[v]:<8} | {bases} (PROBLEME)')
    if occurrences[v] != 2:
        all_two = False

print(f'\nChaque vecteur dans exactement 2 bases : {all_two}')
print(f'Total slots = 36 = 9 bases x 4 = 18 vecteurs x 2 : {sum(occurrences.values()) == 36}')

Vec   | #bases   | Bases contenant ce vecteur
-----------------------------------------------------------------
v0    | 2        | C0 et C1
v1    | 2        | C0 et C4
v2    | 2        | C0 et C2
v3    | 2        | C0 et C6
v4    | 2        | C1 et C4
v5    | 2        | C1 et C7
v6    | 2        | C1 et C3
v7    | 2        | C2 et C3
v8    | 2        | C2 et C5
v9    | 2        | C2 et C6
v10   | 2        | C3 et C5
v11   | 2        | C3 et C7
v12   | 2        | C4 et C8
v13   | 2        | C4 et C5
v14   | 2        | C5 et C8
v15   | 2        | C6 et C7
v16   | 2        | C6 et C8
v17   | 2        | C7 et C8

Chaque vecteur dans exactement 2 bases : True
Total slots = 36 = 9 bases x 4 = 18 vecteurs x 2 : True


### Interpretation : invariant combinatoire verifie

Le tableau confirme que **les 18 vecteurs apparaissent chacun dans exactement 2 des 9 bases**. C'est cette invariance qui rend l'argument de parite possible -- si la repartition etait inegale (par exemple un vecteur dans 3 bases ou un autre dans 1 seule), l'argument echouerait et il pourrait exister une coloration valide.

Cette propriete est exactement ce que la lemme Lean `each_vector_in_two_contexts` enonce dans `KochenSpecker.lean` (cf section 5).

## 5. L'Argument de Parite

Une coloration $c : \text{vecteurs} \to \{0, 1\}$ est dite **valide** si dans chaque base, exactement un vecteur recoit la valeur 1.

### 5.1 Le decompte global

Si une telle coloration existe, le **nombre total de 1 (compte avec multiplicite sur les bases)** est :
$$\sum_{k=0}^{8} (\text{ones dans } C_k) = 9 \cdot 1 = 9$$

puisque chaque base contribue exactement 1.

### 5.2 Le decompte par vecteur

Reordonnons la somme en regroupant par vecteur. Chaque vecteur $v$ contribue $c(v) \cdot (\text{nb de bases contenant } v)$. Avec la propriete d'overlap (chaque vecteur dans 2 bases) :
$$\sum_{k=0}^{8} (\text{ones dans } C_k) = \sum_{v=0}^{17} c(v) \cdot 2 = 2 \cdot \sum_{v=0}^{17} c(v)$$

Ce nombre est **pair**.

### 5.3 La contradiction

Une meme somme vaut a la fois 9 (impair) et 2k (pair). **Contradiction**. Donc aucune coloration valide n'existe.

Verifions ce raisonnement par recherche exhaustive sur les $2^{18} = 262\,144$ colorations possibles.

In [5]:
# Recherche exhaustive : enumerer les 2^18 colorations, compter combien sont valides
n_valid = 0
n_total = 2 ** 18
examples_per_count = {}

for bits in range(n_total):
    c = [(bits >> v) & 1 for v in range(18)]
    # Compter, pour chaque contexte, le nombre de 1
    sums = [sum(c[i] for i in ctx) for ctx in contexts]
    if all(s == 1 for s in sums):
        n_valid += 1
    # Distribution des nombres de contextes "OK" (= sum exactly 1)
    k_ok = sum(1 for s in sums if s == 1)
    examples_per_count[k_ok] = examples_per_count.get(k_ok, 0) + 1

print(f'Total colorations explorees : {n_total:,}')
print(f'Colorations valides (9/9 contextes OK) : {n_valid}')
print()
print(f'Distribution du nombre de contextes "exactement 1" :')
for k in sorted(examples_per_count.keys()):
    pct = 100.0 * examples_per_count[k] / n_total
    bar = '#' * int(pct / 2)
    print(f'  {k}/9 OK : {examples_per_count[k]:>7,} colorations ({pct:5.2f}%) {bar}')

Total colorations explorees : 262,144
Colorations valides (9/9 contextes OK) : 0

Distribution du nombre de contextes "exactement 1" :
  0/9 OK :  30,550 colorations (11.65%) #####
  1/9 OK :  59,760 colorations (22.80%) ###########
  2/9 OK :  66,780 colorations (25.47%) ############
  3/9 OK :  51,768 colorations (19.75%) #########
  4/9 OK :  33,264 colorations (12.69%) ######
  5/9 OK :  13,248 colorations ( 5.05%) ##
  6/9 OK :   5,748 colorations ( 2.19%) #
  7/9 OK :     792 colorations ( 0.30%) 
  8/9 OK :     234 colorations ( 0.09%) 


### Interpretation : verification computationnelle de Kochen-Specker

La recherche exhaustive confirme : **aucune coloration parmi les 262 144 ne satisfait simultanement les 9 contraintes**. C'est la verification "par force brute" du theoreme de Kochen-Specker pour la table Cabello a 18 vecteurs.

On peut aussi observer la distribution : un nombre significatif de colorations satisfait 7 ou 8 contextes sur 9, mais aucune ne franchit la barre de 9/9 -- ce qui aligne avec l'intuition que la contradiction est globale, pas locale.

**Note pedagogique** : la preuve formelle Lean (theorem `kochen_specker` dans `KochenSpecker.lean`) ne procede pas par enumeration mais par argument symbolique (parite), ce qui generalise immediatement a tout n'importe quel ensemble similaire et donne une preuve courte (~10 lignes apres les lemmes prealables).

## 6. Le Port Lean 4 : `KochenSpecker.lean` (Pilier 1)

Le fichier `conway_lean/Conway/KochenSpecker.lean` formalise le theoreme de Kochen-Specker pour la table Cabello. Il sert de **Pilier 1** dans la construction du theoreme de libre arbitre de Conway-Kochen (Epic #1651).

### 6.1 Strategie : formulation abstraite

Plutot que de formaliser les coordonnees R^4 et la verification d'orthogonalite (qui requiert des produits scalaires reels), le scaffold encode la structure combinatoire :
- 18 indices de vecteurs : `VecIdx := Fin 18`
- 9 indices de contextes : `ContextIdx := Fin 9`
- La fonction `contextMembers : ContextIdx -> Fin 4 -> VecIdx` qui realise la table Cabello

L'orthogonalite de chaque contexte est implicite : verifier les produits scalaires est une charge separee, reportee a Pilier 2 (FreeWillTheorem.lean).

### 6.2 La definition `contextMembers`

```lean
/-- An abstract index for the 18 distinct vectors. -/
abbrev VecIdx := Fin 18

/-- An abstract index for the 9 orthogonal bases (contexts). -/
abbrev ContextIdx := Fin 9

/-- Context membership: which 4 vector indices form each orthogonal basis. -/
def contextMembers : ContextIdx -> Fin 4 -> VecIdx
  -- C0: {v0, v1, v2, v3}
  | 0, 0 => 0 | 0, 1 => 1 | 0, 2 => 2 | 0, 3 => 3
  -- C1: {v0, v4, v5, v6}
  | 1, 0 => 0 | 1, 1 => 4 | 1, 2 => 5 | 1, 3 => 6
  -- ... C2 a C8 sur le meme modele
```

Cette definition encode exactement la table verifiee numeriquement en section 3. Le pattern matching sur `(k : Fin 9, i : Fin 4)` couvre les 36 slots de la table.

### 6.3 La coloration valide et le lemme d'overlap

```lean
/-- A {0,1}-coloring of the 18 vectors. -/
def Coloring := VecIdx → Bool

/-- A coloring is valid iff every context has exactly one vector colored true. -/
def IsValidColoring (c : Coloring) : Prop :=
  ∀ k : ContextIdx,
    (∑ i : Fin 4, if c (contextMembers k i) then (1 : ℕ) else 0) = 1

/-- Key property: each of the 18 vectors appears in exactly 2 contexts. -/
lemma each_vector_in_two_contexts (v : VecIdx) :
    (∑ k : ContextIdx, ∑ i : Fin 4,
      if contextMembers k i = v then (1 : ℕ) else 0) = 2 := by
  fin_cases v <;> decide
```

Le lemme `each_vector_in_two_contexts` est l'enonce formel de la propriete combinatoire verifiee en section 4 (chaque vecteur dans exactement 2 bases). La preuve utilise `fin_cases v` pour generer 18 sous-buts (un par vecteur), chacun resolu par le decideur du noyau Lean.

### 6.4 Le theoreme principal et la preuve de parite

```lean
/-- **Kochen-Specker Theorem (18-vector Cabello proof)**.
    There is no valid {0,1}-coloring of the 18 vectors compatible
    with the orthogonality constraint. -/
theorem kochen_specker : ¬ ∃ c : Coloring, IsValidColoring c := by
  rintro ⟨c, hc⟩
  set S : ℕ := ∑ k : ContextIdx, ∑ i : Fin 4,
      if c (contextMembers k i) then (1 : ℕ) else 0 with hS_def
  -- Step 1: S = 9 (each context has exactly one `1`)
  have hS9 : S = 9 := by
    have hsum : S = ∑ _k : ContextIdx, (1 : ℕ) := by
      apply Finset.sum_congr rfl; intro k _; exact hc k
    rw [hsum]; decide
  -- Step 2: rewrite as sum over v, swap with Finset.sum_comm
  have hS_even : S = 2 * (∑ v : VecIdx, if c v then (1 : ℕ) else 0) := by ...
  -- Step 3: 2 | 9 contradiction
  have h2div : 2 ∣ S := ⟨_, hS_even⟩
  rw [hS9] at h2div; omega
```

La preuve complete suit exactement l'argument de la section 5 :

1. Supposer une coloration valide `c`. Sommer sur les 9 contextes : S = 9 (par `IsValidColoring`).
2. Reordonner la double somme via `Finset.sum_comm` : S = 2 * sum_v c(v) (par `each_vector_in_two_contexts`).
3. Donc 2 | 9, contradiction (parite, `omega`).

**Statut** : PROUVE. Les 2 sorrys ont ete elimines en juin 2026 (PR #2019). La cle : `fin_cases v <;> decide` pour le lemme d'overlap + preuve structurelle (pas brute-force `native_decide` qui timeout sur Fin 18).

## 7. Verification : `lake build` + grep sorry

Verifions que le module compile avec 0 sorry (Pilier 1 PROUVE) et que le theoreme de libre arbitre (Pilier 2) est egalement verifie.

In [6]:
import sys
from pathlib import Path

# Cross-platform Lean utilities (Epic #2314)
sys.path.insert(0, str(Path.cwd()))
from lean_notebook_utils import (
    find_lean_project, get_lean_project_path,
    run_lake, count_sorry, is_native_platform,
)

# Backward-compatible aliases for cells later in this notebook
lean_project_win = find_lean_project('conway_lean')
wsl_path = get_lean_project_path('conway_lean')

build_timeout_s = 900  # 15 min

plat = "natif" if is_native_platform() else "WSL"
print(f'Chemin Windows : {lean_project_win}')
print(f'Chemin Lean    : {wsl_path}')
print(f'Plateforme     : {plat}')
print()
print('Verification du module Conway (lake build)...')
print('-' * 60)

rc, out, err = run_lake(wsl_path, 'build Conway', timeout=build_timeout_s)
if out:
    print(out[-1800:] if len(out) > 1800 else out)
if err:
    print('STDERR:', err[-300:])
print()
print(f'Exit code : {rc}')
print('0 = SUCCESS, autre = ECHEC')
if rc == -1:
    print(f'TimeoutExpired apres {build_timeout_s}s (cache mathlib pas prechauffe).')


Chemin Windows : C:\dev\CoursIA\MyIA.AI.Notebooks\SymbolicAI\Lean\conway_lean
Chemin Lean    : /mnt/c/dev/CoursIA/MyIA.AI.Notebooks/SymbolicAI/Lean/conway_lean
Plateforme     : WSL

Verification du module Conway (lake build)...
------------------------------------------------------------



Note: This linter can be disabled with `set_option linter.unusedSimpArgs false`
  List.nil_append

Hint: Omit it from the simp argument list.
  simp only [MacroCell.emptyOfLevel, MacroCell.toCellsAux, ih, L̵i̵s̵t̵.̵a̵p̵p̵e̵n̵d̵_̵n̵i̵l̵,̵ ̵L̵i̵s̵t̵.̵n̵i̵l̵_̵a̵p̵p̵e̵n̵d̵]̵L̲i̲s̲t̲.̲a̲p̲p̲e̲n̲d̲_̲n̲i̲l̲]̲

Note: This linter can be disabled with `set_option linter.unusedSimpArgs false`
⚠ [3347/3350] Replayed Conway.Life.HashlifeMemo
⚠ [3348/3350] Replayed Conway.Life.Pillars
Build completed successfully (3350 jobs).


Exit code : 0
0 = SUCCESS, autre = ECHEC


In [7]:
import os, glob

# Compter les sorrys dans les 2 Piliers de CE notebook : KochenSpecker + FreeWillTheorem
conway_dir = os.path.join(str(lean_project_win), 'Conway')
for fname in ['KochenSpecker.lean', 'FreeWillTheorem.lean']:
    fpath = os.path.join(conway_dir, fname)
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f:
            n = f.read().count('sorry')
        print(f'  {fname}: {n}')
    else:
        print(f'  {fname}: (fichier non trouve)')

print()

# Contexte honnete : lister les AUTRES modules Conway qui contiennent encore "sorry"
# (hors scope de ce notebook, qui ne traite que les Piliers 1+2)
autres = []
for fpath in glob.glob(os.path.join(conway_dir, '**', '*.lean'), recursive=True):
    fname = os.path.relpath(fpath, str(lean_project_win))
    if 'KochenSpecker' in fname or 'FreeWillTheorem' in fname:
        continue
    with open(fpath, 'r', encoding='utf-8') as f:
        if 'sorry' in f.read():
            autres.append(fname)

print('Autres modules Conway contenant encore "sorry" (HORS scope de ce notebook) :')
for f in sorted(autres):
    print(f'  - {f}')
print()
print('Lecture honnete :')
print('  * Piliers 1+2 (KochenSpecker + FreeWillTheorem) : 0 sorry -- formellement PROUVES.')
print('  * Angel.lean / DoomsdayLemmas.lean : "sorry" en commentaire/roadmap (pas de preuve trouee).')
print('  * Life/* (Game of Life, Hashlife) : sorrys de PRODUCTION en cours (Phase 2-3) --')
print('    objectifs P4/P5 intractables + lemmes bridge/scaffolding, documentes dans la roadmap Conway.')
print('  => Ce notebook (Piliers 1+2) ne depend PAS des modules Life/* : ses 2 piliers sont complets.')

  KochenSpecker.lean: 0
  FreeWillTheorem.lean: 0

Autres modules Conway contenant encore "sorry" (HORS scope de ce notebook) :
  - Conway\Angel.lean
  - Conway\DoomsdayLemmas.lean
  - Conway\Life\HashlifeCorrectness.lean
  - Conway\Life\HashlifeMemo.lean
  - Conway\Life\Pillars.lean

Lecture honnete :
  * Piliers 1+2 (KochenSpecker + FreeWillTheorem) : 0 sorry -- formellement PROUVES.
  * Angel.lean / DoomsdayLemmas.lean : "sorry" en commentaire/roadmap (pas de preuve trouee).
  * Life/* (Game of Life, Hashlife) : sorrys de PRODUCTION en cours (Phase 2-3) --
    objectifs P4/P5 intractables + lemmes bridge/scaffolding, documentes dans la roadmap Conway.
  => Ce notebook (Piliers 1+2) ne depend PAS des modules Life/* : ses 2 piliers sont complets.


### Interpretation : Piliers 1+2 PROUVES

| Verification | Resultat | Signification |
|--------------|----------|---------------|
| `lake build Conway` | Exit code 0, ~3340 jobs | Le workspace compile (les `sorry` residuels sont des warnings, pas des erreurs) |
| Sorrys Piliers 1+2 | 0 dans KS + FWT | Les 2 piliers de ce notebook sont complets |
| `KochenSpecker.lean` | 0 sorry | Parite formellement prouvee (PR #2019) |
| `FreeWillTheorem.lean` | 0 sorry | FWT formellement prouve (PR #2026) |

Les 2 piliers mathematiques du theoreme de libre arbitre sont maintenant **entierement prouves** en Lean 4. Le Pilier 1 (KS) via argument de parite (`fin_cases v <;> decide` + `Finset.sum_comm` + `omega`), et le Pilier 2 (FWT) via reduction au Pilier 1 (`SatisfiesSPIN` + `SatisfiesTWIN` + structure MIN).

> **Perimetre.** Les autres modules `Conway/Life/*` (Game of Life, Hashlife) comportent encore des `sorry` de production : ce sont des chantiers de formalisation en cours (Phase 2-3, objectifs P4/P5 intractables + lemmes bridge/scaffolding), hors du perimetre des Piliers 1+2 traites dans ce notebook.

## 8. Pont avec le Theoreme de Libre Arbitre de Conway-Kochen

Le theoreme de Kochen-Specker est le **noyau combinatoire** du theoreme de libre arbitre (Free Will Theorem) de Conway et Kochen (2006, 2009). Ce theoreme affirme :

> Si la reponse d'un experimentateur a un choix de mesure n'est pas une fonction de tout ce qui s'est passe avant, alors la reponse de la particule mesuree non plus.

**Attention pedagogique** : "libre arbitre" chez Conway-Kochen est une **definition mathematique** (la reponse des particules n'est pas une fonction de l'etat anterieur), et **PAS une these philosophique**. Le theoreme demontre rigoureusement qu'un certain type de determinisme est mathematiquement incompatible avec trois axiomes physiques modestes.

Le theoreme repose sur 3 axiomes formalises dans `FreeWillTheorem.lean` :

| Axiome | Enonce intuitif | Formalisation Lean |
|--------|-----------------|---------------------|
| **SPIN** | Pour les particules de spin 1, mesurer le carre du spin selon des axes orthogonaux donne exactement un "1" par base | `SatisfiesSPIN : DeterministicResponse → Prop` |
| **TWIN** | Deux particules entrelacees donnent le meme resultat sur des axes paralleles | `SatisfiesTWIN : TwoParticleResponse → Prop` |
| **MIN** | Les choix d'experimentateurs spatialement separes sont independants | Structural (type signature `TwoParticleResponse`) |

### 8.1 Architecture du port Lean (ETAT ACTUEL, juin 2026)

```
conway_lean/
  Conway/
    KochenSpecker.lean       <-- Pilier 1 : PROUVE (0 sorry, PR #2019)
    FreeWillTheorem.lean      <-- Pilier 2 : PROUVE (0 sorry, PR #2026)
    [Doomsday, LookAndSay,    <-- Autres modules Conway
     Fractran, Nim, Angel,
     Life/* (Phase 2)]
```

### 8.2 Les deux theoremes principaux

**Theoreme 1 (single-particle)** : `fwt_single_particle`
- Hypothese : une fonction de reponse deterministe satisfait SPIN
- Preuve : reduction directe a `kochen_specker` (une telle fonction definit une coloration KS valide, qui n'existe pas)

**Theoreme 2 (two-particle)** : `free_will_theorem`
- Hypothese : un modele deterministe a deux particules satisfait SPIN + TWIN + MIN
- Preuve : par TWIN, Alice et Bob partagent la meme fonction de reponse. Par SPIN(alice), cette fonction satisfait le theoreme a une particule. Contradiction.

### 8.3 Etat de l'Epic #1651

| Pilier | Module | Statut |
|--------|--------|--------|
| 1 | `KochenSpecker.lean` | PROUVE (PR #2019 merge) |
| 2 | `FreeWillTheorem.lean` | PROUVE (PR #2026) |
| 3 | Notebook companion | Ce fichier (Lean-15), mis a jour |

## Exercices

Les concepts vus plus haut (orthogonalite des bases, structure d'overlap, argument de parite)
se pretent a une mise en pratique directe sur les memes objets `vectors`, `contexts`,
`occurrences` et la fonction `dot` deja definis dans ce notebook. Les trois exercices ci-dessous
suivent la progression des sections 3, 4 et 5 : ils reconstruisent pas a pas le coeur de la preuve
de Kochen-Specker. Completez chaque fonction (les cellules s'executent sans erreur tant qu'elles
ne sont pas completees, elles renvoient simplement `None`).


### Exemple guide 1 : Verifier l'orthogonalite d'une base (rappel section 3)

_Resolu par le groupe Matteo Atkinson & Paul Witkowski (contribution PR #2288)._

**Objectif.** Ecrire une fonction qui verifie qu'un contexte (4 indices de vecteurs) forme bien une
base orthogonale de R^4, c'est-a-dire que les 4 vecteurs sont deux a deux orthogonaux.

- `# Indice` : reutilisez `dot(u, v)` ; un contexte de 4 vecteurs comporte 6 paires a tester.
- `# Indice` : deux vecteurs sont orthogonaux quand leur produit scalaire est nul.


In [8]:
def base_est_orthogonale(ctx, vectors):
    # Indice : generer les 6 paires (i, j) avec i < j parmi les 4 indices de ctx
    # Etape 1 : pour chaque paire, calculer dot(vectors[i], vectors[j])
    # Etape 2 : retourner True si tous ces produits scalaires sont nuls, False sinon
    for idx_i in range(len(ctx)):
        for idx_j in range(idx_i + 1, len(ctx)):
            i, j = ctx[idx_i], ctx[idx_j]
            if dot(vectors[i], vectors[j]) != 0:
                return False
    return True

# Test (une fois complete, attendu : True pour chaque base) :
resultat = base_est_orthogonale(contexts[0], vectors)
print(f"base_est_orthogonale(C0) = {resultat} (attendu : True)")
print(f"Verification sur les 9 bases : {all(base_est_orthogonale(ctx, vectors) for ctx in contexts)}")


base_est_orthogonale(C0) = True (attendu : True)
Verification sur les 9 bases : True


### Exemple guide 2 : Compter les slots et leur parite (rappel section 4)

_Resolu par le groupe Matteo Atkinson & Paul Witkowski (contribution PR #2288)._

**Objectif.** L'argument de parite repose sur le decompte des "slots" (paires vecteur-base). Ecrire
une fonction qui calcule le nombre total de slots sur les 9 bases et renvoie aussi sa parite.

- `# Indice` : chaque base contient 4 vecteurs et il y a 9 bases.
- `# Indice` : chaque vecteur apparait dans exactement 2 bases (cf. section 4) ; le total est donc pair.


In [9]:
def slots_totaux_et_parite(contexts):
    # Indice : un "slot" est une occurrence (vecteur, base) ; sommez len(ctx) sur les 9 contextes
    # Etape 1 : total = somme des tailles des 9 contextes
    total = sum(len(ctx) for ctx in contexts)
    # Etape 2 : retourner (total, total % 2)  # parite : 0 = pair, 1 = impair
    return (total, total % 2)

# Test (une fois complete, attendu : (36, 0) -> PAIR) :
res = slots_totaux_et_parite(contexts)
print(f"(total_slots, parite) = {res} (attendu : (36, 0), soit PAIR)")
print(f"Interpretation : 9 bases x 4 vecteurs = {res[0]} slots, {'PAIR' if res[1] == 0 else 'IMPAIR'}")


(total_slots, parite) = (36, 0) (attendu : (36, 0), soit PAIR)
Interpretation : 9 bases x 4 vecteurs = 36 slots, PAIR


### Exemple guide 3 : Reconstituer l'argument de parite (rappel section 5)

_Resolu par le groupe Matteo Atkinson & Paul Witkowski (contribution PR #2288)._

**Objectif.** Sous l'hypothese (fausse, c'est tout l'enjeu) qu'une coloration `{0, 1}` valide existe
-- exactement un vecteur "vrai" par base -- exhiber la contradiction de parite : le decompte "cote
bases" est impair (9) alors que le decompte "cote vecteurs" est forcement pair.

- `# Indice` : cote bases, une coloration valide met exactement un 1 par base -> somme = 9 (impair).
- `# Indice` : cote vecteurs, la meme somme vaut `sum(color[v] * occurrences[v])` ; comme chaque
  `occurrences[v]` vaut 2, cette somme est paire quelles que soient les valeurs `color[v]`.
- `# Indice` : 9 (impair) ne peut pas etre egal a un nombre pair -> aucune coloration valide.


In [10]:
def argument_de_parite(contexts, occurrences):
    # Indice : somme_cote_bases = nombre de bases (un 1 par base dans une coloration valide)
    # Etape 1 : somme_cote_bases = len(contexts)            # = 9, impair
    somme_cote_bases = len(contexts)
    # Etape 2 : la somme cote vecteurs vaut sum(color[v] * occurrences[v]) ; chaque occurrences[v] = 2
    #           donc elle est PAIRE quelle que soit la coloration -> parite_cote_vecteurs = "pair"
    parite_cote_vecteurs = "pair"
    # Etape 3 : retourner (somme_cote_bases, parite_cote_vecteurs)
    return (somme_cote_bases, parite_cote_vecteurs)

# Test (une fois complete, attendu : (9, 'pair') -> 9 impair contredit une somme paire) :
verdict = argument_de_parite(contexts, occurrences)
print(f"(cote_bases, cote_vecteurs) = {verdict}")
print(f"Contradiction : {verdict[0]} est {'impair' if verdict[0] % 2 == 1 else 'pair'} "
      f"(cote bases) mais la somme cote vecteurs est toujours {verdict[1]}.")
print("=> Aucune coloration valide ne peut exister (QED).")


(cote_bases, cote_vecteurs) = (9, 'pair')
Contradiction : 9 est impair (cote bases) mais la somme cote vecteurs est toujours pair.
=> Aucune coloration valide ne peut exister (QED).


---

### Exercice (a completer) : toutes les paires orthogonales de la configuration

Les exemples guides ci-dessus verifient l'orthogonalite *a l'interieur* de chaque base. A votre
tour : comptez le nombre total de paires orthogonales parmi les 18 vecteurs (toutes paires
confondues, pas seulement intra-bases), et comparez-le aux `9 x 6 = 54` paires garanties par
les bases.

- *Indice* : reutilisez `dot(u, v)` et parcourez les paires `(i, j)` avec `i < j` parmi les 18 vecteurs.
- *Etape 1* : enumerer toutes les paires `(i, j)`, `i < j`.
- *Etape 2* : compter celles dont le produit scalaire est nul ; comparer a 54.


In [11]:
# Exercice : compter toutes les paires orthogonales parmi les 18 vecteurs
# TODO etudiant :
#   Etape 1 : parcourir toutes les paires (i, j) avec i < j parmi les 18 vecteurs
#   Etape 2 : compter celles dont dot(vectors[i], vectors[j]) == 0
#   Etape 3 : comparer ce total aux 9 x 6 = 54 paires intra-bases

print("Exercice a completer")


Exercice a completer


## Resume

### Concepts cles

| Concept | Definition | Importance |
|---------|------------|------------|
| **Variables cachees non-contextuelles** | Theorie attribuant des valeurs predeterminees independantes du contexte | Ce que KS exclut |
| **Table Cabello (18 vecteurs)** | 18 vecteurs de R^4 en 9 bases orthogonales, chaque vecteur dans exactement 2 bases | Structure combinatoire minimale |
| **Coloration valide** | Fonction vecteurs -> {0,1} avec exactement un 1 par base | Objet dont on prouve l'inexistence |
| **Argument de parite** | Somme des 1 = 9 (impair) via les bases, mais = 2k (pair) via l'overlap | Contradiction |
| **Theoreme de libre arbitre** | Si experimentateurs libres, alors particules libres | Generalisation de KS via SPIN+TWIN+MIN |

### Architecture du port Lean 4

| Module | Statut | Sorry | Role |
|--------|--------|-------|------|
| `KochenSpecker.lean` | PROUVE | 0 | Parite formelle sur la table Cabello |
| `FreeWillTheorem.lean` | PROUVE | 0 | Reduction FWT -> KS (SPIN+TWIN+MIN) |
| `Conway` (workspace) | Compile | 0 dans KS+FWT ; WIP dans `Life/*` | ~3340 jobs, lake build SUCCESS |

### Verifications realisees dans ce notebook

1. **54 paires orthogonales** verifiees (9 bases x 6 paires)
2. **Invariant d'overlap** : chaque vecteur dans exactement 2 bases
3. **Recherche exhaustive** : 0/262 144 colorations valides
4. **Lake build** : exit code 0 ; 0 sorry dans les Piliers 1+2 (KS + FWT) -- les modules `Life/*` restent un chantier de formalisation en cours

### Prochaine etape

Pour explorer d'autres resultats de John Conway formalises en Lean, voir [Lean-14b-Conway-Game-of-Life-Lean](Lean-14b-Conway-Game-of-Life-Lean.ipynb). Pour la robustesse des reseaux de neurones, voir [Lean-11-TorchLean](Lean-11-TorchLean.ipynb).

## 9. Pour aller plus loin

### References historiques

1. **S. Kochen, E. P. Specker**, "The Problem of Hidden Variables in Quantum Mechanics", J. Math. Mech. 17 (1967), 59-87. La preuve originale a 117 vecteurs en R^3.

2. **A. Peres**, "Two simple proofs of the Kochen-Specker theorem", J. Phys. A 24 (1991), L175-L178. Preuve a 33 vecteurs.

3. **M. Kernaghan**, "Bell-Kochen-Specker theorem for 20 vectors", J. Phys. A 27 (1994), L829-L830.

4. **A. Cabello, J. M. Estebaranz, G. Garcia-Alcaine**, "Bell-Kochen-Specker theorem: A proof with 18 vectors", Phys. Lett. A 212 (1996), 183-187. **La table utilisee dans ce notebook**.

5. **J. H. Conway, S. Kochen**, "The Free Will Theorem", Found. Phys. 36 (2006), 1441-1473. arXiv:quant-ph/0604079.

6. **J. H. Conway, S. Kochen**, "The Strong Free Will Theorem", Notices AMS 56 (2009), 226-232.

### Implications philosophiques

Le theoreme KS et sa generalisation au theoreme de libre arbitre demolissent une classe entiere d'interpretations realistes locales de la mecanique quantique :

- **Bohm-De Broglie (pilot wave)** : sauve si on accepte la contextualite (les valeurs cachees dependent du contexte de mesure)
- **Many-Worlds (Everett)** : non concerne (pas de variables cachees)
- **GRW (collapse)** : non concerne
- **Local hidden variables** : exclu

### Liens vers d'autres notebooks

- [Lean-12 Sensitivity](Lean-12-Sensitivity-Theorem.ipynb) : autre theoreme combinatoire avec preuve Lean compacte (Huang 2019)
- [Lean-14 Conway Tribute](Lean-14b-Conway-Game-of-Life-Lean.ipynb) : hommage Conway, Game of Life
- [Lean-11 TorchLean](Lean-11-TorchLean.ipynb) : verification formelle pour ML

### Exercices

1. Verifier manuellement que la base C2 = (v7, v8, v2, v9) est orthogonale en calculant les 6 produits scalaires.
2. Pour la table de Peres a 33 vecteurs (R^3), construire un tableau similaire. Quelle est la structure d'overlap ?
3. Lire `conway_lean/Conway/FreeWillTheorem.lean` et tracer la chaine de reduction : `free_will_theorem` -> `fwt_single_particle` -> `kochen_specker`. Combien d'etapes ?
4. (Avance) Etendre `FreeWillTheorem.lean` avec des coordonnees R^4 explicites et une preuve d'orthogonalite pour chaque contexte (actuellement implicite dans `contextMembers`).

---

**Navigation** : [<< Lean-14 Conway Tribute](Lean-14b-Conway-Game-of-Life-Lean.ipynb) | [Index](README.md)